# Create and Fill Bank Monitoring Database

This notebook renders the bank monitoring DDL with Russian comments, creates the SQLite database through the project pipeline, and previews the first rows from each generated table.

In [34]:
from contextlib import redirect_stdout
import io
import os
from pathlib import Path
import sqlite3
import sys
from IPython.display import Markdown, display
import pandas as pd

from src.config import PROJECT_ROOT, RAW_DATA_DIR
from src.dataset import (  # noqa: E402
    build_database,
    render_ddl,
)


COMMENT_STYLE = "yaml" # "inline" or "yaml"
COMMENT_VARIANT = "technical" # "short" or "business" or "technical"
PREVIEW_ROWS = 5
DATABASE_NAME = "bank_transaction_monitoring" # "bank_transaction_monitoring" or "sakila"
SCHEMA_MAPPING_PATH = os.path.join(RAW_DATA_DIR, DATABASE_NAME, "schema_mapping.json")

In [35]:
ddl = render_ddl(
    comment_style=COMMENT_STYLE,
    comment_variant=COMMENT_VARIANT,
    database_name=DATABASE_NAME 
)

print(ddl)

-- table: btm_cst
-- source_table: BANK_CUST
-- description: Таблица измерения клиентов; ключ соединения с таблицами счетов хранится в поле c01.
-- columns:
--   c01: Целочисленный идентификатор клиента; используется как внешний ключ в a01 и s01.  # source: customer_id
--   c02: Текстовое имя клиента без фамилии; пример значения: Oliver.  # source: customer_name
--   c03: Текстовый адрес клиента, обычно строка с номером дома и городом.  # source: Address
--   c04: Трехсимвольный региональный код; типовые значения: CA, MN, NY, SFO.  # source: state_code
--   c05: Телефон в виде строки из цифр, чтобы сохранять ведущие нули и формат исходной системы.  # source: Telephone
CREATE TABLE btm_cst (
    c01 INTEGER,
    c02 TEXT,
    c03 TEXT,
    c04 TEXT,
    c05 TEXT
);

-- table: btm_cex
-- source_table: BANK_CUSTOMER_EXPORT
-- description: Таблица контрольной выгрузки клиентов с текстовыми типами, близкая по структуре к btm_cst.
-- columns:
--   x01: Идентификатор клиента в текстовом форма

In [36]:
db_path = build_database(
    comment_style=COMMENT_STYLE,
    comment_variant=COMMENT_VARIANT,
    database_name=DATABASE_NAME,
    overwrite=True,
)

print(f"SQLite database created: {db_path}")

schema_mapping = pd.read_json(SCHEMA_MAPPING_PATH, orient="index")
table_names = schema_mapping.index.tolist()

with sqlite3.connect(db_path) as connection:
    for table_name in table_names:
        row_count = connection.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        display(Markdown(f"## `{table_name}` ({row_count} rows)"))
        display(pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT {PREVIEW_ROWS}", connection))

SQLite database created: /home/user/cursor_projects/vanna-sql/data/processed/bank_transaction_monitoring/bank_transaction_monitoring_yaml_technical.sqlite.db


## `btm_cst` (50 rows)

,c01,c02,c03,c04,c05
0,123001,Oliver,"200-1, Emeryville",CA,1897614000
1,123002,George,"201-2, New Brighton",MN,1897614137
2,123003,Harry,"202-3, Walnut Creek",NY,1897614274
3,123004,Jack,"203-4, Concord",TX,1897614411
4,123005,Jacob,"204-5, Mission Dist",WA,1897614548


## `btm_cex` (50 rows)

,x01,x02,x03,x04,x05
0,123001,Oliver,"200-1, Emeryville",CA,1897614000
1,123002,George,"201-2, New Brighton",MN,1897614137
2,123003,Harry,"202-3, Walnut Creek",NY,1897614274
3,123004,Jack,"203-4, Concord",TX,1897614411
4,123005,Jacob,"204-5, Mission Dist",WA,1897614548


...

,t01,t02,t03,t04,t05
0,4000-1900-3000,-1000.0,ATM withdrawal,NY,2020-01-05
1,5000-1700-5000,-2500.0,POS-Walmart,TX,2020-01-10
2,9000-1700-7000-4300,-4000.0,UPI transfer,WA,2020-01-15
3,4000-1901-3001,-5500.0,Bankers cheque,IL,2020-01-20
4,5000-1701-5001,7000.0,Net banking,AZ,2020-01-25


## `btm_msg` (50 rows)

,m01,m02,m03
0,Adhoc,All bank branches are closed due to a national...,mobile
1,Transaction Limit,Only limited withdrawals per card are allowed ...,mobile
2,Card Security,Confirm recent card activity if the transactio...,sms
3,Balance Alert,Your account balance crossed the configured mo...,email
4,Rate Update,Interest rate has been updated for selected de...,email


## `btm_rate` (100 rows)

,i01,i02,i03,i04
0,SAVINGS,0.036,01,2020
1,RECURRING DEPOSITS,0.051,01,2020
2,PRIVILEGED_INTEREST_RATE,0.066,01,2020
3,Credit Card,0.081,01,2020
4,SAVINGS,0.037,02,2020
